In [ ]:
%pip install matplotlib

from __future__ import annotations

import random
import string
import matplotlib.pyplot as plt


class ProbingHashTable:
    """Hash table demonstrating linear vs. quadratic probing."""

    _DEFAULT_CAPACITY: int = 11
    _PROBE_MODES: tuple[str, str] = ("linear", "quadratic")
    _DEFAULT_PROBING_MODE: str = _PROBE_MODES[0]

    def __init__(
        self, capacity: int = _DEFAULT_CAPACITY, mode: str = _DEFAULT_PROBING_MODE
    ):
        """Initialize the hash table with the given capacity and probing mode."""
        if mode not in self._PROBE_MODES:
            mode = self._DEFAULT_PROBING_MODE
        self._capacity = capacity
        self._mode = mode
        self._table: list[str | None] = [None] * capacity
        self._size = 0

    def _hash(self, key: int) -> int:
        """Convert a nonnegative integer to an array index."""
        return key % self._capacity

    def _probe(self, position: int, attempt: int) -> int:
        """Return the index to probe on the given attempt."""
        if self._mode == "linear":
            return (position + attempt) % self._capacity
        return (position + attempt * attempt) % self._capacity

    def load_factor(self) -> float:
        """Return the current load factor."""
        return self._size / self._capacity

    def size(self) -> int:
        """Return number of stored entries."""
        return self._size

    def capacity(self) -> int:
        """Return table capacity."""
        return self._capacity

    def insert(self, value: str) -> list[int]:
        """
        Insert value. Return the list of probed indices.

        If probing fails to find an empty slot, returns the probes attempted
        and leaves the table unchanged.
        """
        probes: list[int] = []

        if self._size < self._capacity:
            key = abs(hash(value))
            index = self._hash(key)

            i = 0
            insertion_successful = False

            while i < self._capacity and not insertion_successful:
                probe_index = self._probe(index, i)
                probes.append(probe_index)

                slot = self._table[probe_index]
                insertion_successful = slot is None or slot == value

                if insertion_successful:
                    self._table[probe_index] = value
                    if slot is None:
                        self._size += 1

                i += 1

        return probes

    def contains(self, value: str) -> bool:
        """Return True if value is stored in the table."""
        key = abs(hash(value))
        index = self._hash(key)

        found = False
        i = 0

        while i < self._capacity and not found:
            probe_index = self._probe(index, i)
            slot = self._table[probe_index]
            if slot is not None:
                found = slot == value
            i += 1

        return found


def random_string(length: int = 8) -> str:
    """Return a random alphabetic string."""
    alphabet = string.ascii_letters
    chars: list[str] = []
    for _ in range(length):
        chars.append(random.choice(alphabet))
    return "".join(chars)


def insertion_succeeded(
    table_before_size: int,
    table_after_size: int,
    value_already_present: bool,
) -> bool:
    """
    Determine whether an insertion attempt should count as a successful insertion
    that advances the load factor.

    We only count it when the table size actually increases.
    """
    return (not value_already_present) and (table_after_size == table_before_size + 1)


def run_one_trial(
    mode: str,
    capacity: int,
    max_failed_attempts_at_same_size: int,
    string_length: int,
) -> tuple[list[int], list[int], int]:
    """
    Run one trial for one probing mode.

    Returns a tuple:
        successful_probe_counts:
            probe count for each successful insertion
        failed_attempts_before_each_success:
            number of failed insertion attempts that occurred before each
            successful insertion
        terminal_failed_attempts:
            number of failed attempts at the final size before we gave up
    """
    table = ProbingHashTable(capacity=capacity, mode=mode)

    successful_probe_counts: list[int] = []
    failed_attempts_before_each_success: list[int] = []

    failed_at_current_size = 0

    while table.size() < capacity and failed_at_current_size < max_failed_attempts_at_same_size:
        value = random_string(string_length)

        # Avoid counting duplicate insertions as successful insertions
        already_present = table.contains(value)
        size_before = table.size()
        probes = table.insert(value)
        size_after = table.size()

        if insertion_succeeded(size_before, size_after, already_present):
            successful_probe_counts.append(len(probes))
            failed_attempts_before_each_success.append(failed_at_current_size)
            failed_at_current_size = 0
        else:
            failed_at_current_size += 1

    terminal_failed_attempts = failed_at_current_size
    return successful_probe_counts, failed_attempts_before_each_success, terminal_failed_attempts


def simulate(
    mode: str,
    capacity: int,
    trials: int,
    max_failed_attempts_at_same_size: int,
    string_length: int,
) -> tuple[list[float], list[float], list[float], list[float]]:
    """
    Run many trials for one probing mode.

    Returns four parallel lists:
        load_factors
        avg_probes_per_success
        avg_failures_before_success
        reach_fraction

    For position j:
        load factor = (j + 1) / capacity

    reach_fraction tells us what fraction of trials actually reached that load factor.
    This becomes important for quadratic probing near high load factors.
    """
    probe_sums: list[int] = [0] * capacity
    failure_sums: list[int] = [0] * capacity
    counts: list[int] = [0] * capacity

    for _ in range(trials):
        success_probes, failures_before_success, _terminal_failures = run_one_trial(
            mode=mode,
            capacity=capacity,
            max_failed_attempts_at_same_size=max_failed_attempts_at_same_size,
            string_length=string_length,
        )

        i = 0
        while i < len(success_probes):
            probe_sums[i] += success_probes[i]
            failure_sums[i] += failures_before_success[i]
            counts[i] += 1
            i += 1

    load_factors: list[float] = []
    avg_probes_per_success: list[float] = []
    avg_failures_before_success: list[float] = []
    reach_fraction: list[float] = []

    i = 0
    while i < capacity:
        load_factor = (i + 1) / capacity
        load_factors.append(load_factor)

        if counts[i] > 0:
            avg_probes_per_success.append(probe_sums[i] / counts[i])
            avg_failures_before_success.append(failure_sums[i] / counts[i])
            reach_fraction.append(counts[i] / trials)
        else:
            avg_probes_per_success.append(float("nan"))
            avg_failures_before_success.append(float("nan"))
            reach_fraction.append(0.0)

        i += 1

    return (
        load_factors,
        avg_probes_per_success,
        avg_failures_before_success,
        reach_fraction,
    )


def print_results_table(
    mode: str,
    load_factors: list[float],
    avg_probes: list[float],
    avg_failures: list[float],
    reach_fraction: list[float],
) -> None:
    """Print a readable table of the results."""
    print()
    print(mode.upper())
    print(
        f"{'load':>8} {'avg probes':>12} {'avg fails':>12} {'reach frac':>12}"
    )
    print("-" * 48)

    i = 0
    while i < len(load_factors):
        lf = load_factors[i]
        ap = avg_probes[i]
        af = avg_failures[i]
        rf = reach_fraction[i]

        avg_probes_text = f"{ap:.3f}" if ap == ap else "N/A"
        avg_failures_text = f"{af:.3f}" if af == af else "N/A"

        print(
            f"{lf:8.3f} {avg_probes_text:>12} {avg_failures_text:>12} {rf:12.3f}"
        )
        i += 1


def plot_probe_curves(
    linear_data: tuple[list[float], list[float], list[float], list[float]],
    quadratic_data: tuple[list[float], list[float], list[float], list[float]],
) -> None:
    """Plot average probes per successful insertion."""
    linear_load, linear_avg_probes, _, _ = linear_data
    quad_load, quad_avg_probes, _, _ = quadratic_data

    plt.figure(figsize=(10, 6))
    plt.plot(linear_load, linear_avg_probes, marker="o", label="Linear probing")
    plt.plot(quad_load, quad_avg_probes, marker="s", label="Quadratic probing")
    plt.xlabel("Load factor")
    plt.ylabel("Average probes per successful insertion")
    plt.title("Average probes per insertion vs. load factor")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_reach_fraction(
    linear_data: tuple[list[float], list[float], list[float], list[float]],
    quadratic_data: tuple[list[float], list[float], list[float], list[float]],
) -> None:
    """
    Plot fraction of trials that reached each load factor.

    This is especially useful for quadratic probing.
    """
    linear_load, _, _, linear_reach = linear_data
    quad_load, _, _, quad_reach = quadratic_data

    plt.figure(figsize=(10, 6))
    plt.plot(linear_load, linear_reach, marker="o", label="Linear probing")
    plt.plot(quad_load, quad_reach, marker="s", label="Quadratic probing")
    plt.xlabel("Load factor")
    plt.ylabel("Fraction of trials that reached load factor")
    plt.title("How far each probing method gets")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def main() -> None:
    """
    Main driver.

    Suggested settings:
    - capacity should be prime for quadratic probing
    - more trials gives smoother averages
    - max_failed_attempts_at_same_size controls when we decide a method
      is 'struggling' at a given load factor
    """
    random.seed(271)

    capacity = 101
    trials = 400
    max_failed_attempts_at_same_size = 200
    string_length = 8

    linear_data = simulate(
        mode="linear",
        capacity=capacity,
        trials=trials,
        max_failed_attempts_at_same_size=max_failed_attempts_at_same_size,
        string_length=string_length,
    )

    quadratic_data = simulate(
        mode="quadratic",
        capacity=capacity,
        trials=trials,
        max_failed_attempts_at_same_size=max_failed_attempts_at_same_size,
        string_length=string_length,
    )

    print_results_table("linear", *linear_data)
    print_results_table("quadratic", *quadratic_data)

    plot_probe_curves(linear_data, quadratic_data)
    plot_reach_fraction(linear_data, quadratic_data)


if __name__ == "__main__":
    main()

: 